# 3.5 Exercise: what does an image cost, and what does it buy?

Notebook 3.1 drew what parameters and data buy as curves. Notebook 3.2 put a mixture of
experts beside a dense head that uses as many parameters per image, on seven kinds of
image. Here you measure both yourself, on the same seven kinds of image, and you measure
one thing more for every model: what one image costs it.

At the end you can:

- read a **Pareto front** of accuracy against the cost of one image from runs you
  measured, name the runs on it, and say why "the best trial" needs a second measure:
  its cost;
- read a curve over data and width: does a wider head pull ahead as the data grows,
  and what does a smaller head give up;
- put scaling into a search space: data and width as axes, the cost logged on every
  trial, and a control of equal compute beside an architecture of your own.

## What is asked of you

1. **Part 1: run this notebook as it is.** A baseline, a curve over data and width, a
   mixture of experts with its control of equal compute, and a short search; every run
   drawn on one front. You change nothing: on a slow machine the notebook shortens
   itself (`QUICK`, after E1). You read what comes out and answer the questions for yourself. Run
   Part 1 before `make study LESSON=3`: its search measures what a trial costs on your
   machine, and the study's budget is computed from that.
2. **Part 2: one study of your own**, in the folder [`exercises/`](exercises/): one idea,
   a design that can test it, a search, and, when it adds an option, a control of equal
   compute beside it. The class spreads over the open questions at the end of this
   notebook, and builds on each other's ideas.
3. **Hand in your runs** before the next lesson, with one command: `make share LESSON=3`.
   It hands in Part 1 and every study of yours, each run with the design script it ran.

Every experiment below says what it cost on a recent laptop (Apple M-series, CPU); Part 1
took 12 minutes there. Part 2 is a study of `MINUTES` on your machine, 30 to
start, and you may give it more of the homework's about 3 hours. A slow machine runs Part 1 with `QUICK`: after E1 the notebook says how long the rest
takes on yours, and turns `QUICK` on when that is more than half an hour.

## The data: seven kinds of image, two vectors each

The mix holds every image of seven datasets: textures (DTD), satellite tiles (EuroSAT),
flowers, dishes (Food-101), traffic signs (GTSRB), galaxies and blood cells, 321 classes
in all. Every image was run once through the pretrained encoder `dinov2-small` (lesson
1), and the cache keeps two of its vectors, 384 numbers each:

- the **CLS** vector, the encoder's one summary of the image, the one `ImageVectorizer`
  gave you in lesson 1;
- the **patch mean**: the encoder also gives one vector per patch of 14 x 14 pixels, and
  the mean of those 256 vectors is a second summary of the image.

A model here reads one or both, and gives a score per class. The domain of an image is
in the data, but no model is given it.

A **validation split**, 10% of every class, is cut from train once and is the same for
every run: every score in this notebook is on it. Test is left alone, for the end of
the course.

A run can train on less than every train row. **`per_class`** is a cap: the run takes at
most that many train images of every class. The classes are very unequal, from 26
images of one flower to 2,397 of one kind of blood cell, and a class with fewer images
than the cap stays whole: cutting it further would leave a head almost nothing to learn
it from. So a smaller cap pulls the classes towards equal sizes, and the mix of domains
shifts with it; the table under Setup shows how. The images under a cap of 80 are a
subset of those under a cap of 250. Every run logs its cap and the train rows that cap
left it: `per_class` and `n_train`, in every table of runs below.

## What one image costs

Lesson 1 counted the parameters of a head. A parameter stored is not a parameter used:
the mixture of experts of 3.2 stores eight experts and runs one per image. Two
measures say what one image costs, and every run in this lesson logs both beside
`n_params`:

- **FLOPs per image**: the floating-point operations of one forward pass, per image. A
  `Linear(384, 64)` does a multiply and an add per weight, 2 x 384 x 64 = 49,152 FLOPs.
  A mixture of experts counts its router and the one expert that ran, not the experts it
  stores.
- **latency per image**: the measured time of one forward pass over a batch of 256
  images, divided by 256, in microseconds, on one CPU thread. FLOPs count arithmetic;
  latency is what the clock says, and the two need not agree (3.2.6).

These FLOPs count the head. The encoder costs billions of FLOPs per image (3.1.7), but
it ran once per image, when the cache was built: every run here reads the stored
vectors, so the head is the cost that differs between runs, as for a service that stores
the vectors once and runs its heads on them. In this notebook **compute** means FLOPs per
image: a control of equal compute does as many FLOPs per image.

## The Pareto front

Most pairs of runs do not say which is better: one scores higher and costs more. A run
is on the **Pareto front** when no other run is at least as accurate at no more FLOPs per
image and better on one of the two. Runs of one width under three caps do the same FLOPs,
so only the most accurate of them can be on it. With FLOPs on the x-axis, better is up
and to the left. "The best trial" picks the highest score and ignores its cost; the
front shows what each step in accuracy costs.

## Setup

The data, the trainer and the helpers (`run`, `table`, `plot_front`, `search`) come from
[`exercises/harness.py`](exercises/harness.py), the design `Mix` from
[`exercises/designs.py`](exercises/designs.py): the same code your study uses in Part 2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from exercises.designs import DESIGN, Config, control_width
from exercises.harness import (
    DOMAINS,
    EPOCHS,
    HARNESS,
    HUES,
    LINEAR,
    NOTEBOOK,
    PER_CLASS_ALL,
    Linear,
    cut,
    from_study,
    load,
    plain_ticks,
    plot_front,
    run,
    search,
    stored_rows,
    table,
)
from notebooktester import param

from hpo.measure import Architecture
from hpo.record import StudyLog
from hpo.search import Budget

WIDTHS = [16, 64, 256]  # the hidden widths of E2
# the caps of E2: at most 80 or 250 train images of every class, or every train row
CAPS = param([80, 250, PER_CLASS_ALL], test=[80])
N_TRIALS = param(12, test=2)  # the trials of E4
QUICK_TRIALS = param(4, test=2)  # the trials of E4 with QUICK (see after E1)
COLOURS = dict(zip(["E1", "E2", "E3", "E4"], HUES, strict=False))
# every choice of the design as Config gives it, and the trainer's: every train row of
# dinov2-small, as flat vectors
TRAINER = {
    "dataset": "mix7",
    "encoder": "dinov2-small",
    "per_class": PER_CLASS_ALL,
    "drop_domain": "none",
    "form": "flat",
    "lr": 1e-3,
    "epochs": EPOCHS,
}
PLAIN = {**vars(Config()), **TRAINER}


def build(config: dict, data):
    """The model of the design that `config` chooses, at the sizes of `data`."""
    return DESIGN.build(
        config,
        dim=data.dims["dim"],
        height=data.dims["height"],
        breadth=data.dims["breadth"],
        n_classes=len(data.classes),
    )


mix = load("dinov2-small")
train_rows, valid_rows = cut("dinov2-small")


def per_domain(rows):
    """How many of `rows` each domain holds."""
    return np.bincount(mix.domain[rows], minlength=len(DOMAINS))


overview = pd.DataFrame(
    {
        "classes": [
            len(np.unique(mix.label[mix.domain == d])) for d in range(len(DOMAINS))
        ],
        "validation": per_domain(valid_rows),
        "train": per_domain(train_rows),
        "train, per_class 250": per_domain(cut("dinov2-small", 250)[0]),
        "train, per_class 80": per_domain(cut("dinov2-small", 80)[0]),
    },
    index=DOMAINS,
)
overview.loc["all"] = overview.sum()
display(overview)
print(f"every image: {mix.embed.shape[1:]}, the CLS vector and the patch mean")

Food-101 holds almost half the train rows, DTD and the flowers few: a model good at food
and traffic signs scores high on accuracy, whatever it does on textures.

The two columns on the right show what a cap does to that mix. At 80 images per class
DTD is whole and the flowers nearly, Food-101 falls from 46% of the train rows to 36%,
and the three domains of few, large classes (EuroSAT, the galaxies, the blood cells)
fall from 31% of the rows to 10%. The validation split is the same under every cap, so
a run under a small cap is scored on a mix of domains its train rows do not match.

# Part 1: the notebook as handed out

How a run trains, in every experiment: Adam at a learning rate of 1e-3, batches of 256.
Every run gets the same steps per epoch, enough for all 148,090 train rows, so a run
under a cap of 80, on 22,602 rows, sees each of them six or seven times per epoch. A run
stops once its validation accuracy has not improved for `PATIENCE` (10) epochs in a row,
or after `EPOCHS` (150), and keeps the parameters of its best epoch, the **kept epoch**.

## E1: the baseline, first

A linear layer on the CLS vector, every train row: `Linear(384, 321)`. It is what every
later number is read against, the reference here and in your own study.

**Cost:** half a minute, plus the first download of the mix.

In [ ]:
e1 = run(
    "linear",
    {**LINEAR, **TRAINER},
    "baseline",
    "E1",
    make=lambda config, data: Linear(data.dims["dim"], len(data.classes)),
)
table([e1])

## How long the rest of Part 1 takes here

E2 to E4 train about 55 times as long as E1 did: E1 is one linear layer, the rest are
two dozen heads, several of them on every row. The cell below turns E1's time into
minutes for the rest. A laptop built for battery life rather than for compute can be
twenty or thirty times slower than the one this notebook was written on, and then the
rest runs for hours.

`QUICK = True` keeps it short: E2 trains its runs under a cap of 80 and of 250 and takes
its runs on every row from a stored full run of Part 1 (`part1_stored.json`), E3 shows the stored
mixture and its control instead of training them, and E4 runs 4 trials instead of 12.
A stored run is named "(stored)" in every table and plot. It ran from the same seed on
another machine, so its accuracy and FLOPs are close to what yours would be; its latency
is that machine's clock, not yours. The cell turns `QUICK` on when the rest would take
more than half an hour; set it yourself to overrule it. E4 runs even then, because its
trials measure what one trial costs on this machine, and `make study` turns your
`MINUTES` into trials at that cost.

In [ ]:
# E2 to E4 take about this many times E1's time: as handed out, and with QUICK
REST_FULL, REST_QUICK = 55, 15
rest = e1["minutes"] * REST_FULL
QUICK = rest > 30  # or set True or False yourself
print(
    f"E1 took {e1['minutes'] * 60:.0f}s here: the rest of Part 1 takes about "
    f"{rest:.0f} minutes, about {e1['minutes'] * REST_QUICK:.0f} with QUICK"
)
print(f"QUICK = {QUICK}")

A linear layer costs two FLOPs per weight: 2 x 384 x 321 = 246,528 per image. A head with
a hidden layer can cost less, because the hidden layer is narrow.

## E2: a curve over data and width

**Question.** Does a wider head pull ahead as the data grows? A head with one hidden
layer of width 16, 64 or 256, under a cap of 80 and of 250 images per class and on every
train row: 22,602, 51,782 and 148,090 rows.

The head is the design of [`exercises/designs.py`](exercises/designs.py) with every
option off: the CLS vector, one hidden layer, the classifier. Open it in atlas before you
run it:

```bash
atlas-code-nightly view notebooks/3_scaling_and_search/exercises
```

<svg width="720" height="80" viewBox="0 0 720 80" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="12">
<defs><marker id="e2a" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<rect width="720" height="80" fill="white"/>
<text x="4" y="44" fill="#222">image</text>
<line x1="46" y1="40" x2="128" y2="40" stroke="#444" stroke-width="1.5" marker-end="url(#e2a)"/>
<text x="88" y="32" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 2, 384)</text>
<rect x="130" y="22" width="90" height="36" rx="6" fill="#eeeeee" stroke="#555"/>
<text x="175" y="44" text-anchor="middle" fill="#222">choose CLS</text>
<line x1="220" y1="40" x2="290" y2="40" stroke="#444" stroke-width="1.5" marker-end="url(#e2a)"/>
<text x="255" y="32" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 384)</text>
<rect x="292" y="22" width="150" height="36" rx="6" fill="#cfe3f3" stroke="#555"/>
<text x="367" y="44" text-anchor="middle" fill="#222">Linear(384, w), GELU</text>
<line x1="442" y1="40" x2="508" y2="40" stroke="#444" stroke-width="1.5" marker-end="url(#e2a)"/>
<text x="475" y="32" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, w)</text>
<rect x="510" y="22" width="110" height="36" rx="6" fill="#cfe3f3" stroke="#555"/>
<text x="565" y="44" text-anchor="middle" fill="#222">Linear(w, 321)</text>
<line x1="620" y1="40" x2="700" y2="40" stroke="#444" stroke-width="1.5" marker-end="url(#e2a)"/>
<text x="660" y="32" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 321)</text>
</svg>

`B` is the batch; row 0 of an image's two vectors is its CLS vector.

The cost of one image is 2 x (384 x w + w x 321) FLOPs: 22,560 at width 16, 360,960 at
width 256. Width 16 costs a tenth of the linear layer; width 256 costs more than it.

**Setup.** Nine runs, one per width and cap, each trained until it settles, from the
same seed. A run under a cap is not short of steps, only of distinct images. A cap
changes two things at once: how many rows a run has, and the mix of domains among them.
Each point is one run; another seed moves it by a few tenths of a point, and by more on
few rows.

Under a cap of 80, which width scores highest? On every row? Does the gap between
width 16 and width 256 grow or shrink as the cap grows?

**Cost:** 3 minutes, most of it the runs on every train row. With `QUICK` those three
are the stored ones.

In [ ]:
def cap_name(per_class: int) -> str:
    """A cap in the name of a run: `c80`, or `all` for every train row."""
    return "all" if per_class == PER_CLASS_ALL else f"c{per_class}"


e2 = [
    run(
        f"w{width} {cap_name(per_class)}",
        {**PLAIN, "width": width, "per_class": per_class},
        "hypothesis",
        "E2",
        build,
    )
    for per_class in CAPS
    if not (QUICK and per_class == PER_CLASS_ALL)
    for width in WIDTHS
]
if QUICK:
    e2 += stored_rows("E2", per_class=PER_CLASS_ALL)
table(e2)

In [ ]:
fig, (curve, settle) = plt.subplots(1, 2, figsize=(13, 4))
hues = dict(zip(WIDTHS, ["#E69F00", "#009E73", "#0072B2"], strict=True))
styles = {80: ":", 250: "--", PER_CLASS_ALL: "-"}
# the train rows each cap left, from small to large
n_train = dict(sorted({r["per_class"]: r["n_train"] for r in e2}.items()))
for width in WIDTHS:
    mine = [r for r in e2 if r["width"] == width]
    curve.plot(
        [r["n_train"] for r in mine],
        [r["accuracy"] for r in mine],
        "o-",
        color=hues[width],
        label=f"width {width}",
    )
    for r in mine:
        epochs = np.arange(1, len(r["curve"]) + 1)
        settle.plot(epochs, r["curve"], styles[r["per_class"]], color=hues[width], lw=1)
        settle.scatter(
            [r["kept_epoch"]],
            [r["curve"][r["kept_epoch"] - 1]],
            color=hues[width],
            s=18,
        )
curve.axhline(
    e1["accuracy"], color="black", ls="--", lw=1, label="linear, every row (E1)"
)
curve.set_xscale("log")
curve.set_xticks(
    list(n_train.values()),
    [
        f"{n:,}\n" + ("every row" if c == PER_CLASS_ALL else f"per_class {c}")
        for c, n in n_train.items()
    ],
)
curve.minorticks_off()
curve.set_xlabel("train rows (log scale), and the cap that left them")
curve.set_ylabel("validation accuracy")
curve.set_title(
    "the curve over data and width: one run per point", loc="left", fontsize=10
)
curve.legend(fontsize=8)
settle.set_xscale("log")
settle.set_xlabel("epoch (log scale; the same steps under every cap)")
settle.set_ylabel("validation accuracy")
settle.set_title(
    "every run per epoch; dot = kept epoch; dotted cap 80, dashed cap 250, solid every row",
    loc="left",
    fontsize=10,
)
for ax in (curve, settle):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

by = {(r["width"], r["per_class"]): r["accuracy"] for r in e2}
for per_class, n in n_train.items():
    gap = 100 * (by[WIDTHS[-1], per_class] - by[WIDTHS[0], per_class])
    print(
        f"per_class {per_class:>4d}, {n:>7,} train rows: "
        f"width {WIDTHS[-1]} - width {WIDTHS[0]} = {gap:+.1f} points"
    )
part1 = [e1, *e2]

Read the two panels:

1. **Has every run settled?** In the right panel, read each line before its dot: still
   climbing, flat, or turned down.
2. **Does the wider head pull ahead as the data grows?** Read the printed gaps, width 256
   minus width 16: do they grow or shrink as the cap grows? On every row width 256 scores
   0.861 against the linear layer's 0.868. As in 3.1.6, a head on top of frozen vectors
   does not learn the images, so it gets little more than what a linear layer reads from
   them: what does that leave more data to add to width 256, and what to width 16?
3. **Against the linear layer.** The dashed line in the left panel is E1. The rule of
   thumb: a head with more parameters than its data needs overfits, and trained past its
   best epoch its validation accuracy stops rising or falls: the line of width 256 on
   every row lies flat after its dot in the right panel. Which lines under a cap turn
   down? Keeping the
   best epoch is early stopping, the only regularisation these runs have. With it, width
   256 on every row ends about level with the linear layer at 1.5 times its FLOPs: the
   bigger head buys nothing here and costs compute.
4. **What does a smaller head give up?** Read the table: how many points does width 16
   give up against the linear layer, at a tenth of its FLOPs, and against width 64?
5. **Where would you put the ranges of a search?** Where the curve still changes. Does
   every width still rise from a cap of 250 to every row? Then the axis `per_class` runs
   to every row. Over width, on every row, 16 to 64 buys 7 points and 64 to 256 buys 2:
   the bend lies near 64, so a log axis from 16 to 256 puts trials on both sides of it,
   where one from 128 up would spend them on the flat part.
6. **What else did the cap change?** Under a cap of 80 a run has fewer rows and another
   mix of domains: Food-101 gave up most of its images, DTD none. The curve does not say
   which of the two a loss of accuracy comes from. The recall per class does, and every
   run logs it: `make optuna-dashboard` shows it per run under "User Attributes",
   `metrics`.

## E3: a mixture of experts on the front

3.2's mixture of experts: 8 experts, each a head with one hidden layer of width 8, and a
router, `Linear(384, 8)`, that runs one of them per image (the option `moe` of the
design). In 3.2 its gain came from the split the router learned. Beside it runs its
**control of equal compute**, the plain head that does as many FLOPs per image: the cell
below counts the mixture's FLOPs before anything is trained, and `control_width` gives
the matching width.

In 3.2 the mixture gained where the budget was tight. Its control here has width 12:
does that lie on the steep part of E2's curve over width or on the flat part, and what
does that leave the mixture to gain?

**Cost:** 4 minutes; the router's gather and scatter make every step of the
mixture slower than its FLOPs say. With `QUICK` the cell shows the stored run of both.

In [ ]:
EXPERT_WIDTH, N_EXPERTS = 8, 8
moe_config = {**PLAIN, "moe": True, "width": EXPERT_WIDTH, "n_experts": N_EXPERTS}
data = HARNESS.data(PLAIN)
moe_flops = Architecture.of(build(moe_config, data), data.example).flops_per_image
control = control_width(moe_flops, d_in=384, n_classes=321)
control_config = {**PLAIN, "width": control}
for name, config in [
    (f"MoE 8 x {EXPERT_WIDTH}", moe_config),
    (f"dense {control}", control_config),
]:
    arch = Architecture.of(build(config, data), data.example)
    print(
        f"{name:>10s}: {arch.flops_per_image:>7,} FLOPs per image, {arch.n_params:>8,} parameters stored"
    )

In [ ]:
if QUICK:
    e3 = stored_rows("E3")
else:
    e3 = [
        run(f"MoE 8 x {EXPERT_WIDTH}", moe_config, "hypothesis", "E3", build),
        run(f"dense {control}", control_config, "control", "E3", build),
    ]
display(table(e3))
part1 += e3
on = plot_front(part1, "every run of E1 to E3: accuracy against what one image costs")
print("on the front:", ", ".join(r["name"] for r in on))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for experiment in ["E1", "E2", "E3"]:
    mine = [r for r in part1 if r["experiment"] == experiment]
    ax.scatter(
        [r["flops"] for r in mine],
        [r["latency_us"] for r in mine],
        color=COLOURS[experiment],
        s=30,
        label=experiment,
    )
for r in part1:
    if r["per_class"] == PER_CLASS_ALL:
        ax.annotate(
            r["name"],
            (r["flops"], r["latency_us"]),
            xytext=(4, 2),
            textcoords="offset points",
            fontsize=8,
        )
ax.set_xscale("log")
ax.set_yscale("log")
plain_ticks(ax.xaxis)
plain_ticks(ax.yaxis)
ax.set_xlabel("FLOPs per image (log scale)")
ax.set_ylabel("latency per image, µs (log scale)")
ax.set_title(
    "does the clock agree with the FLOPs? one thread, batches of 256; "
    "unlabelled: E2 under a cap",
    loc="left",
    fontsize=10,
)
ax.legend(fontsize=8, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Read the front and the clock. A ring marks a run on the front, and the grey steps join
them:

1. **Which runs are on the front?** Name them, from cheap to dear. For every run off the
   front, which run on it is at least as accurate at no more FLOPs?
2. **Where did the mixture land,** against its control and against width 16 on every
   row? What do the seven experts it stores and does not run buy?
3. **What does the clock say?** Read the mixture's latency against its control's:
   sending every image to its expert and putting the scores back is work the FLOPs do not
   count. The linear layer, one matrix product, is no slower than the narrowest heads at
   ten times the FLOPs of width 16: fewer FLOPs is not automatically faster (3.2.6), and
   a few hundredths of a µs between equal heads is the clock's noise. On a front drawn
   on latency, the mixture would lie far off: the cost on the x-axis decides the front.

## E4: a starter search, on the same front

**Question.** A search spends its trials over a space; where on the front do its trials
land? The space is `SPACE` of [`exercises/study.py`](exercises/study.py), read from the
file: the starter space below.

| axis | values | what it turns |
|---|---|---|
| `width` | 16 to 256 | the hidden layers: parameters and FLOPs |
| `depth` | 1 or 2 | hidden layers |
| `per_class` | 20 to 2,397 (every row) | data: the most train images of every class |
| `input` | CLS, patch mean, both | what the head reads: both doubles the first layer |
| `dropout` | 0 to 0.3 | regularisation: while training, that share of the hidden numbers is set to zero at random, so the head cannot lean on any one |

A cap does not change a trial's FLOPs per image: a trial under a small cap lies lower,
not further left. What it saves is labels, and training time for a narrow head, which
settles in fewer epochs on fewer rows; the front shows neither.

The search samples at random, not with 3.3's TPE: TPE on accuracy alone draws its trials
towards the most accurate heads, and a front needs trials at the cheap end too. It prunes
nothing: a pruner stops a trial on its first
epochs, and the small heads, which learn slowest at first, are the ones the front asks
about.

Will the trial with the highest accuracy be on the front? A cap changes no FLOPs:
which kind of trial do you expect to land off the front?

**Cost:** 5 minutes for 12 trials; 4 trials with `QUICK`.

In [ ]:
STARTER_SPACE = from_study("SPACE")
e4 = search(STARTER_SPACE, QUICK_TRIALS if QUICK else N_TRIALS, build, "E4")
pd.DataFrame(e4).sort_values("accuracy", ascending=False).drop(
    columns="experiment"
).style.format(
    {
        "accuracy": "{:.3f}",
        "flops": "{:,.0f}",
        "latency_us": "{:.2f}",
        "n_train": "{:,.0f}",
        "dropout": "{:.2f}",
    }
).hide(axis="index")

In [ ]:
on = plot_front(
    part1 + e4, "Part 1 and the starter search: accuracy against what one image costs"
)
print("on the front:", ", ".join(r["name"] for r in on))
best = max(e4, key=lambda r: r["accuracy"])
print(
    f"the best trial: {best['name']}, accuracy {best['accuracy']:.3f}, "
    f"{best['flops']:,} FLOPs per image"
)

1. **Is the best trial on the front?** Only if no other run is at least as accurate at
   no more FLOPs and better on one of the two.
2. **With 30,000 FLOPs per image to spend, which run would you take?** And with
   100,000? A run on the front by a few tenths of a point, or by a few hundred FLOPs
   (in the run behind this text, width 16 on every row lies half a point above the
   mixture), is on it by a hair: another seed or a
   slightly other width moves it. Read where the front bends and which kind of run lies
   there, not which run is on it by a hair.
3. **Which trials are off the front?** What do they have in common, in the table?

# Part 2: your own study

The loop is the one of lessons 1 and 2, on lesson 3's design: its nine steps, with their
commands, are in [`docs/your-study.md`](../../docs/your-study.md), the same in every
lesson. You work in [`exercises/`](exercises/):

| file | what it is | what you do with it |
|---|---|---|
| [`exercises/designs.py`](exercises/designs.py) | `Mix`, one typed method per stage, and `Head` inside it; the options are `kernel` and `moe` on `Mix`, `skip` and `gate` on `Head` | read it with atlas; add a stage when your idea needs another block |
| [`exercises/study.py`](exercises/study.py) | six settings at the top (`STUDY`, `IDEA`, `EXPECTS`, `OBJECTIVE`, `SPACE`, `MINUTES`) and the code that runs the study | fill in the settings, then `make study LESSON=3` |
| [`exercises/harness.py`](exercises/harness.py) | the mix, `per_class`, `drop_domain`, `encoder`, `form`, the trainer that stops when a run settles, the linear baseline, and this notebook's helpers | nothing: it is given |

**A control of equal compute** is for a question that adds an option to the model.
Without the plain head that does as many FLOPs per image beside it, a gain of your design
may be one of more FLOPs. FLOPs, not parameters:
a kernel stores few weights and does many FLOPs, since each weight is used at every
position of the grid.

1. Count your design's FLOPs before you train, as E3 did:
   `Architecture.of(model, data.example).flops_per_image`;
   `control_width(flops, d_in, n_classes=321)` is the matching plain head's width, with
   `d_in` the length of the vector it reads: the encoder's width (`data.dims["dim"]`),
   twice that for `both`.
2. Search your option as `Choice([False, True])` beside `width`, over a range that holds
   that width: the trials without the option are the control, and step 3 reads the
   others against them at the same FLOPs. Fix `per_class` (small, on a slow laptop) and every
   axis your question does not study, so that the control differs from your design only
   in the option and in the width that matches.
3. Draw your study's front with the two apart, in a cell of this notebook:

   ```python
   from exercises.harness import rows_of

   plot_front(rows_of("lesson3-study-<your study>", by="<your option>"), "my study")
   ```

   Your option pays where its trials lie above the others at the same FLOPs.

For 3.2's mixture: its 17,424 FLOPs give `control_width` 12, so `"width": Int(8, 64,
log=True)` beside `"moe": Choice([False, True])` holds both (with `moe` on, `width` is
each expert's width), and `by="moe"` draws the trials with and without it.

A question that adds no option (the curve over data, how small you can go, small against
base, leaving a domain out) compares its runs along its own axis, on the same front as
Part 1's runs, with the baseline.

## The open questions

Every question below is one bet a study can test. The class spreads over them, so that
the next lesson opens on a map of what scale and architecture bought, not a leaderboard;
a bet that did not pay is a result on that map.

- **Can a hidden layer beat the linear layer?** On every row, width 256 ends about level
  with it. Does a head wider than the 384 numbers of the vector, with `dropout`, pass it,
  and at how many FLOPs per image?
- **The curve over data.** E2 measured three sizes: a cap of 80, a cap of 250 and every
  row. Where does the curve over `per_class` bend, at the same cap for every width, and
  for every domain alike? A cap above a domain's largest class changes nothing for it.
- **How small can you go?** Where does the front bend at its cheap end, and which axis
  buys the most accuracy per FLOP there: width, input or depth?
- **Experts against their control.** E3 measured one mixture. Does a mixture of more
  experts, or wider ones, stay above its control of equal compute?
- **A kernel on the vector.** `form="grid"` folds every vector into a grid of 16 x 24,
  `(B, 2, 16, 24)`, as lesson 2 folded a beat: numbers 24 apart in the vector become
  neighbours, and the option `kernel` reads the grid with a 2D kernel and a pool. A kernel
  can pay for two reasons. *Neighbours*: in a beat, neighbouring samples belong together;
  are two neighbours in this grid more correlated than any two numbers of the vector? You
  can measure that on the cache before you train anything. *Sharing*: a kernel uses the
  same few weights at every position, so it stores far fewer parameters than a dense
  layer, and does many FLOPs. Does that sharing help without neighbours, for instance under
  a small cap on the data? Read it against a plain head of the same FLOPs, and against
  one of the same parameters.
- **CLS, patch mean, or both?** Does the patch mean help some domains more than others
  (textures, galaxies)? Every run logs the recall of every class, `<domain>/<class>`:
  `make optuna-dashboard` shows it per trial under "User Attributes", `metrics`.
  Is `both`, which doubles the first layer, worth its FLOPs against a wider head on CLS?
- **Deeper heads.** What does `depth` 2 to 4 buy against a wide head of one layer at the
  same FLOPs, and do `skip` and `gate` change it? With what regularisation (`dropout`)
  or normalisation (`norm="layer"`, lesson 2's LayerNorm), and how much data? A highway (lesson 2) is a stage you can add.
- **Small against base.** `encoder="dinov2-base"` gives vectors of 768 numbers from a
  larger encoder. Does a better encoder buy more than a wider head, at the FLOPs of the
  head? The encoder's own cost per image is not in these FLOPs: say what that leaves out.
- **Leave a domain out.** `drop_domain` takes one domain out of train and validation, so
  that run is scored on six domains and another on seven. Compare the six shared domains
  in both, from the recall per class: does the rest score higher without it, and which
  domain costs the others most?

A question of your own is welcome.

## Two of the steps, for lesson 3

- **Step 4, the settings** of `study.py`: `STUDY` is your first name and what it studies,
  `"sam-kernel-on-the-fold"`. `IDEA` says what you investigate in your own words; its
  `builds_on` names the idea yours starts from, Part 1 (`Idea.handed_out(3).name`) or an
  idea of the class by its name, as `make dashboard LESSON=3` shows them. `EXPECTS` is what you expect, and why, written before
  you run. `OBJECTIVE` is the metric the search maximises, `accuracy` or
  `macro_recall`. `SPACE` searches the axes your question needs, with the control of
  equal compute among its trials. `MINUTES` is your budget; the cell below says what it
  buys.
- **Step 6, reading your result:** your option against its control on your front, drawn
  as above, and the winner against the baseline in points, which `make study LESSON=3`
  prints.

In [ ]:
MINUTES = from_study("MINUTES")
PART1 = StudyLog.load(name=NOTEBOOK)
seconds = PART1.seconds_per_trial(epochs=EPOCHS)
budget = Budget.at(
    minutes=MINUTES,
    seconds_per_trial=seconds,
    measured_on=PART1.name,
    epochs=EPOCHS,
    repeats=HARNESS.repeats,
)
print(
    f"a trial of Part 1 took {seconds:.0f} s here (the median); MINUTES = {MINUTES} "
    f"of study.py buys {budget.n_trials} trials"
)

`make study LESSON=3` turns `MINUTES` into this number of trials, at what a trial of
Part 1 cost, and stops the search when `MINUTES` are up, even when its trials run slower:
the trials that ran are kept, and the next `make study` runs the ones left. A trial of
your space can cost more or less than one of the starter space: a run under a small cap
can settle in fewer epochs, a mixture of experts or a kernel takes longer per step. The
first trials of your study tell you (`make optuna-dashboard` shows their
`train_seconds`): when they are slower, fewer of them fit in `MINUTES`, so fix a smaller
cap or give the study more minutes.